# **SIMPLE SEVERITY-BASED HOTSPOT DETECTION**

In [ ]:
import pandas as pd
import kagglehub
import folium
from folium.plugins import MarkerCluster
from geopy.distance import geodesic

# SAFE CHUNK LOADING 
print("📥 Loading UK Accident dataset...")

dataset_path = kagglehub.dataset_download("silicon99/dft-accident-data")
file_path = dataset_path + "/Accidents0515.csv"

chunk_list = []

for chunk in pd.read_csv(
    file_path,
    chunksize=40000,
    usecols=["Latitude", "Longitude", "Accident_Severity"],
    on_bad_lines="skip",
    low_memory=False
):
    chunk = chunk.dropna(subset=["Latitude", "Longitude"])
    chunk_list.append(chunk)

df = pd.concat(chunk_list, ignore_index=True)
print("✔ Loaded:", df.shape)


# BALANCE THE DATASET
print("⚖️ Balancing dataset (equal Fatal, Serious, Slight)...")

fatal_df = df[df["Accident_Severity"] == 1]
serious_df = df[df["Accident_Severity"] == 2]
slight_df = df[df["Accident_Severity"] == 3]

min_size = min(len(fatal_df), len(serious_df), len(slight_df))

fatal_df_bal = fatal_df.sample(min_size, random_state=42)
serious_df_bal = serious_df.sample(min_size, random_state=42)
slight_df_bal = slight_df.sample(min_size, random_state=42)

df = pd.concat([fatal_df_bal, serious_df_bal, slight_df_bal], ignore_index=True)

print("✔ Balanced dataset counts:")
print(df["Accident_Severity"].value_counts())


# DEFINE SEVERITY COLORS
def get_color(sev):
    if sev == 1:
        return "red"       # Fatal
    elif sev == 2:
        return "orange"    # Serious
    else:
        return "green"     # Slight


# CREATE MAP & PLOT POINTS
print("🗺️ Creating accident severity map...")

severity_map = folium.Map(
    location=[df["Latitude"].mean(), df["Longitude"].mean()],
    zoom_start=6
)

marker_cluster = MarkerCluster().add_to(severity_map)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=5,                      
        color=get_color(row["Accident_Severity"]),
        fill=True,
        fill_color=get_color(row["Accident_Severity"]),
        fill_opacity=0.95,              # DARKER
    ).add_to(marker_cluster)

severity_map.save("SEVERITY_MAP.html")
print("✔ Saved base map: SEVERITY_MAP.html")


# RISK CHECKER + DYNAMIC 3 KM CIRCLE
def get_risk(lat, lon, radius_km=3):
    fatal = 0
    serious = 0
    slight = 0

    # Add 3 km black boundary circle
    folium.Circle(
        location=[lat, lon],
        radius=radius_km * 1000,
        color="black",
        weight=2,
        fill=True,
        fill_color="lightblue",
        fill_opacity=0.30
    ).add_to(severity_map)

    # Add user marker
    folium.Marker(
        location=[lat, lon],
        popup="User Location",
        icon=folium.Icon(color="blue")
    ).add_to(severity_map)

    # Count accidents in 3 km radius
    for _, row in df.iterrows():
        dist = geodesic((lat, lon), (row["Latitude"], row["Longitude"])).km

        if dist <= radius_km:
            if row["Accident_Severity"] == 1:
                fatal += 1
            elif row["Accident_Severity"] == 2:
                serious += 1
            else:
                slight += 1

    counts = {
        "FATAL": fatal,
        "SERIOUS": serious,
        "SLIGHT": slight
    }

    verdict = max(counts, key=counts.get)

    return {
        "Fatal Accidents": fatal,
        "Serious Accidents": serious,
        "Slight Accidents": slight,
        "Final Verdict": verdict
    }


📥 Loading UK Accident dataset...
✔ Loaded: (1780515, 3)
⚖️ Balancing dataset (equal Fatal, Serious, Slight)...
✔ Balanced dataset counts:
1    22998
2    22998
3    22998
Name: Accident_Severity, dtype: int64
🗺️ Creating accident severity map...
✔ Saved base map: SEVERITY_MAP.html


In [ ]:
# GETUSER INPUT
print("\n🔵 Enter Location to Check Accident Risk")

try:
    user_lat = float(input("📍 Enter Latitude: "))
    user_lon = float(input("📍 Enter Longitude: "))
except:
    print("❌ Invalid input. Please enter numeric latitude & longitude.")
    raise SystemExit

print("\n⏳ Analyzing 3 km area around your location...\n")

result = get_risk(user_lat, user_lon)

severity_map.save("USER_RISK_RESULT_MAP.html")

# PRINT OUTPUT
print("=============================================")
print("          🚦 ACCIDENT RISK REPORT            ")
print("=============================================")
print(f"📌 User Location: ({user_lat}, {user_lon})")
print("---------------------------------------------")
print(f"🔴 Fatal Accidents   : {result['Fatal Accidents']}")
print(f"🟠 Serious Accidents : {result['Serious Accidents']}")
print(f"🟢 Slight Accidents  : {result['Slight Accidents']}")
print("---------------------------------------------")
print(f"📢 FINAL RISK VERDICT: {result['Final Verdict']}")
print("---------------------------------------------")
print("🗺️ Map saved as: USER_RISK_RESULT_MAP.html")
print("=============================================")


🔵 Enter Location to Check Accident Risk

⏳ Analyzing 3 km area around your location...

          🚦 ACCIDENT RISK REPORT            
📌 User Location: (52.4862, -0.1585)
---------------------------------------------
🔴 Fatal Accidents   : 2
🟠 Serious Accidents : 1
🟢 Slight Accidents  : 1
---------------------------------------------
📢 FINAL RISK VERDICT: FATAL
---------------------------------------------
🗺️ Map saved as: USER_RISK_RESULT_MAP.html
